In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    roc_auc_score,
    average_precision_score
)

print("Day 5 libraries imported successfully")

Day 5 libraries imported successfully


In [7]:
df = pd.read_csv("../data/frix_features_day1.csv")

print("Dataset loaded:", df.shape)
df[["step", "type", "amount", "isFraud"]].head()

Dataset loaded: (1048575, 20)


,step,type,amount,isFraud
0,1,PAYMENT,9839.64,0
1,1,PAYMENT,1864.28,0
2,1,TRANSFER,181.00,1
3,1,CASH_OUT,181.00,1
4,1,PAYMENT,11668.14,0


In [8]:
print("Minimum step:", df["step"].min())
print("Maximum step:", df["step"].max())
print("Unique steps:", df["step"].nunique())

df["step"].describe()

Minimum step: 1
Maximum step: 95
Unique steps: 95


count    1.048575e+06
mean     2.696617e+01
std      1.562325e+01
min      1.000000e+00
25%      1.500000e+01
50%      2.000000e+01
75%      3.900000e+01
max      9.500000e+01
Name: step, dtype: float64

In [9]:
train_df = df[df["step"] <= 80].copy()
test_df = df[df["step"] > 80].copy()

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

print("Train fraud rate:", train_df["isFraud"].mean() * 100)
print("Test fraud rate:", test_df["isFraud"].mean() * 100)

Train shape: (1030645, 20)
Test shape: (17930, 20)
Train fraud rate: 0.09430987391390828
Test fraud rate: 0.9481316229782488


In [10]:
feature_cols = [
    "amount",
    "oldbalanceOrg",
    "newbalanceOrig",
    "oldbalanceDest",
    "newbalanceDest",
    "origin_balance_error",
    "dest_balance_error",
    "is_high_risk_type",
    "sender_txn_count",
    "receiver_txn_count",
    "sender_emptied_account",
    "is_large_amount",
    "risk_score_v1"
]

X_train = train_df[feature_cols]
y_train = train_df["isFraud"]

X_test = test_df[feature_cols]
y_test = test_df["isFraud"]

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("Train fraud count:", y_train.sum())
print("Test fraud count:", y_test.sum())

X_train: (1030645, 13)
X_test: (17930, 13)
Train fraud count: 972
Test fraud count: 170


In [11]:
rf_time_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_time_model.fit(X_train, y_train)

print("Time-based Random Forest trained successfully")

Time-based Random Forest trained successfully


In [12]:
y_pred_time = rf_time_model.predict(X_test)
y_prob_time = rf_time_model.predict_proba(X_test)[:, 1]

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_time))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_time))

print("ROC-AUC:", roc_auc_score(y_test, y_prob_time))
print("PR-AUC:", average_precision_score(y_test, y_prob_time))

Confusion Matrix:
[[17760     0]
 [    0   170]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     17760
           1       1.00      1.00      1.00       170

    accuracy                           1.00     17930
   macro avg       1.00      1.00      1.00     17930
weighted avg       1.00      1.00      1.00     17930

ROC-AUC: 0.9999999999999999
PR-AUC: 0.9999999999999999


In [13]:
day5_results = pd.DataFrame([
    {
        "model": "Day 5 Random Forest",
        "validation_type": "Time-based split",
        "train_steps": "1-80",
        "test_steps": "81-95",
        "precision_fraud": 1.00,
        "recall_fraud": 1.00,
        "false_positives": 0,
        "false_negatives": 0,
        "roc_auc": roc_auc_score(y_test, y_prob_time),
        "pr_auc": average_precision_score(y_test, y_prob_time)
    }
])

day5_results

,model,validation_type,train_steps,test_steps,precision_fraud,recall_fraud,false_positives,false_negatives,roc_auc,pr_auc
0,Day 5 Random Forest,Time-based split,1-80,81-95,1.0,1.0,0,0,1.0,1.0


In [14]:
strict_feature_cols = [
    "amount",
    "oldbalanceOrg",
    "newbalanceOrig",
    "oldbalanceDest",
    "newbalanceDest",
    "is_high_risk_type",
    "sender_txn_count",
    "receiver_txn_count"
]

X_train_strict = train_df[strict_feature_cols]
X_test_strict = test_df[strict_feature_cols]

print("Strict X_train:", X_train_strict.shape)
print("Strict X_test:", X_test_strict.shape)

Strict X_train: (1030645, 8)
Strict X_test: (17930, 8)


In [15]:
rf_strict_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_strict_model.fit(X_train_strict, y_train)

print("Strict time-based Random Forest trained successfully")

Strict time-based Random Forest trained successfully


In [16]:
y_pred_strict = rf_strict_model.predict(X_test_strict)
y_prob_strict = rf_strict_model.predict_proba(X_test_strict)[:, 1]

print("Strict Model Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_strict))

print("\nStrict Model Classification Report:")
print(classification_report(y_test, y_pred_strict))

print("Strict Model ROC-AUC:", roc_auc_score(y_test, y_prob_strict))
print("Strict Model PR-AUC:", average_precision_score(y_test, y_prob_strict))

Strict Model Confusion Matrix:
[[17577   183]
 [    0   170]]

Strict Model Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.99      0.99     17760
           1       0.48      1.00      0.65       170

    accuracy                           0.99     17930
   macro avg       0.74      0.99      0.82     17930
weighted avg       1.00      0.99      0.99     17930

Strict Model ROC-AUC: 0.9994127583465819
Strict Model PR-AUC: 0.9497036690441237


In [17]:
day5_results = pd.DataFrame([
    {
        "model": "Time RF - Full Features",
        "validation_type": "Time-based split",
        "feature_set": "Day 1 engineered features",
        "train_steps": "1-80",
        "test_steps": "81-95",
        "precision_fraud": 1.00,
        "recall_fraud": 1.00,
        "false_positives": 0,
        "false_negatives": 0,
        "roc_auc": roc_auc_score(y_test, y_prob_time),
        "pr_auc": average_precision_score(y_test, y_prob_time)
    },
    {
        "model": "Time RF - Strict Features",
        "validation_type": "Time-based split",
        "feature_set": "Reduced leakage-risk feature set",
        "train_steps": "1-80",
        "test_steps": "81-95",
        "precision_fraud": 0.48,
        "recall_fraud": 1.00,
        "false_positives": 183,
        "false_negatives": 0,
        "roc_auc": roc_auc_score(y_test, y_prob_strict),
        "pr_auc": average_precision_score(y_test, y_prob_strict)
    }
])

day5_results

,model,validation_type,feature_set,train_steps,test_steps,precision_fraud,recall_fraud,false_positives,false_negatives,roc_auc,pr_auc
0,Time RF - Full Features,Time-based split,Day 1 engineered features,1-80,81-95,1.00,1.0,0,0,1.000000,1.000000
1,Time RF - Strict Features,Time-based split,Reduced leakage-risk feature set,1-80,81-95,0.48,1.0,183,0,0.999413,0.949704


In [18]:
day5_results.to_csv("../models/day5_time_based_validation_results.csv", index=False)

print("Day 5 time-based validation results saved successfully")

Day 5 time-based validation results saved successfully


In [19]:
import os

os.listdir("../models")

['day2_model_comparison.csv',
 'day4_feature_importance.csv',
 'day4_graph_model_comparison.csv',
 'day5_time_based_validation_results.csv',
 'random_forest_fraud_model_day2.joblib',
 'random_forest_graph_model_day4.joblib']